---
title: "CDEaDPP 2025.2 - Trabalho Final"
subtitle: "Ciência de Dados e Estatística aplicada a Direito e Políticas Públicas"
author: "Cleuler Barbosa das Neves"
date: today
format:
  html:
    toc: true
    toc-depth: 3
    toc-title: "Sumário"
    number-sections: true
    code-fold: false
    code-tools: true
    theme: cosmo
  pdf:
    toc: true
    number-sections: true
    colorlinks: true
---


In [ ]:
#| label: setup
#| include: false

library(tidyverse)
library(knitr)
library(kableExtra)
library(ggplot2)
library(pwr)

# Configurações gerais
theme_set(theme_minimal(base_size = 12))
options(digits = 4)
set.seed(2025)


# Quadro de Variáveis da Pesquisa

## Dicionário de Dados


In [ ]:
#| label: tbl-dicionario
#| tbl-cap: "Dicionário de dados da pesquisa"

# Criar dicionário de dados
dicionario <- tibble(
  Nome = c(
    "id",
    "dias_preso",
    "idade_fato",
    "tipo_penal",
    "tipo_acao",
    "reincidencia"
  ),
  Finalidade = c(
    "Identificador único do caso",
    "Número de dias em que o réu permaneceu preso",
    "Idade do réu em anos completos na data do fato",
    "Classificação do tipo penal imputado",
    "Tipo de ação processual",
    "Se houve reincidência criminal"
  ),
  Tipo = c(
    "Identificador",
    "Quantitativa contínua",
    "Quantitativa discreta",
    "Categórica nominal",
    "Categórica nominal",
    "Categórica binária"
  ),
  Exemplo = c(
    "001, 002, 003...",
    "15, 30, 120...",
    "18, 25, 42...",
    "Furto, Roubo, Tráfico...",
    "Pública, Privada",
    "Sim, Não"
  )
)

# Exibir tabela formatada
kable(dicionario, 
      align = c("l", "l", "l", "l"),
      booktabs = TRUE) %>%
  kable_styling(
    bootstrap_options = c("striped", "hover", "condensed"),
    full_width = FALSE,
    position = "left"
  ) %>%
  column_spec(1, bold = TRUE, width = "3cm") %>%
  column_spec(2, width = "6cm") %>%
  column_spec(3, width = "4cm") %>%
  column_spec(4, width = "4cm")


### Descrição das Variáveis

**Variáveis Quantitativas:**

-   **dias_preso**: Número de dias que o réu permaneceu preso preventivamente ou após condenação
-   **idade_fato**: Idade do réu em anos completos na data em que ocorreu o fato criminoso

**Variáveis Categóricas:**

-   **tipo_penal**: Classificação do crime (Furto, Roubo, Tráfico de Drogas, Homicídio, Outros)
-   **tipo_acao**: Natureza da ação processual (Pública ou Privada)
-   **reincidencia**: Indica se o réu é reincidente (Sim ou Não)

# Dados Simulados


In [ ]:
#| label: criar-dados-simulados
#| echo: true

# Criar dados simulados para 100 observações
n <- 100

dados <- tibble(
  id = sprintf("%03d", 1:n),
  
  # Variável quantitativa: dias preso (distribuição assimétrica)
  dias_preso = round(rexp(n, rate = 1/60) + runif(n, 0, 10)),
  
  # Variável quantitativa: idade no fato (distribuição aproximadamente normal)
  idade_fato = round(rnorm(n, mean = 32, sd = 12)),
  
  # Variável categórica: tipo penal
  tipo_penal = sample(
    c("Furto", "Roubo", "Tráfico", "Homicídio", "Outros"),
    n,
    replace = TRUE,
    prob = c(0.25, 0.20, 0.30, 0.10, 0.15)
  ),
  
  # Variável categórica: tipo de ação
  tipo_acao = sample(
    c("Pública", "Privada"),
    n,
    replace = TRUE,
    prob = c(0.90, 0.10)
  ),
  
  # Variável categórica binária: reincidência
  reincidencia = sample(
    c("Sim", "Não"),
    n,
    replace = TRUE,
    prob = c(0.35, 0.65)
  )
) %>%
  # Ajustar idades para valores realistas (18 a 75 anos)
  mutate(idade_fato = pmax(18, pmin(75, idade_fato)))

# Visualizar primeiras observações
head(dados, 10)


## Salvando os Dados


In [ ]:
#| label: salvar-dados
#| echo: true

# Criar diretório se não existir
if (!dir.exists("out")) {
  dir.create("out")
}

# Salvar em CSV
write_csv(dados, "out/dados_pesquisa.csv")

# Salvar em RDS
saveRDS(dados, "out/dados_pesquisa.rds")

cat("Dados salvos com sucesso em:
")
cat("  - out/dados_pesquisa.csv
")
cat("  - out/dados_pesquisa.rds
")


# Análise Exploratória de Dados (AED) - Univariada

## Análise da Variável: Dias Preso


In [ ]:
#| label: fig-dias-preso
#| fig-cap: "Distribuição de frequência dos dias de prisão"
#| fig-width: 10
#| fig-height: 6

# Criar gráficos
p1 <- ggplot(dados, aes(x = dias_preso)) +
  geom_histogram(bins = 30, fill = "steelblue", alpha = 0.7, color = "white") +
  labs(title = "Histograma: Dias de Prisão",
       x = "Dias preso",
       y = "Frequência") +
  theme_minimal()

p2 <- ggplot(dados, aes(x = dias_preso)) +
  geom_boxplot(fill = "steelblue", alpha = 0.7) +
  labs(title = "Boxplot: Dias de Prisão",
       x = "Dias preso") +
  theme_minimal() +
  theme(axis.text.y = element_blank(),
        axis.ticks.y = element_blank())

library(patchwork)
p1 / p2


### Medidas Resumo: Dias Preso


In [ ]:
#| label: resumo-dias-preso

# Estatísticas descritivas
resumo_dias <- dados %>%
  summarise(
    N = n(),
    Média = mean(dias_preso),
    Mediana = median(dias_preso),
    `Desvio Padrão` = sd(dias_preso),
    Mínimo = min(dias_preso),
    `Q1 (25%)` = quantile(dias_preso, 0.25),
    `Q3 (75%)` = quantile(dias_preso, 0.75),
    Máximo = max(dias_preso),
    Assimetria = moments::skewness(dias_preso),
    Curtose = moments::kurtosis(dias_preso)
  )

kable(t(resumo_dias), 
      col.names = "Valor",
      digits = 2) %>%
  kable_styling(bootstrap_options = c("striped", "hover"))


**Interpretação:**

\[Descrever os padrões observados na distribuição de dias preso, identificar outliers se houver, comentar sobre assimetria e dispersão dos dados\]

## Análise da Variável: Idade no Fato


In [ ]:
#| label: fig-idade-fato
#| fig-cap: "Distribuição de frequência da idade no momento do fato"
#| fig-width: 10
#| fig-height: 6

p3 <- ggplot(dados, aes(x = idade_fato)) +
  geom_histogram(bins = 25, fill = "darkgreen", alpha = 0.7, color = "white") +
  geom_density(aes(y = after_stat(density) * n), color = "red", linewidth = 1) +
  labs(title = "Histograma: Idade no Fato",
       x = "Idade (anos)",
       y = "Frequência") +
  theme_minimal()

p4 <- ggplot(dados, aes(x = idade_fato)) +
  geom_boxplot(fill = "darkgreen", alpha = 0.7) +
  labs(title = "Boxplot: Idade no Fato",
       x = "Idade (anos)") +
  theme_minimal() +
  theme(axis.text.y = element_blank(),
        axis.ticks.y = element_blank())

p3 / p4


### Medidas Resumo: Idade no Fato


In [ ]:
#| label: resumo-idade

resumo_idade <- dados %>%
  summarise(
    N = n(),
    Média = mean(idade_fato),
    Mediana = median(idade_fato),
    `Desvio Padrão` = sd(idade_fato),
    Mínimo = min(idade_fato),
    `Q1 (25%)` = quantile(idade_fato, 0.25),
    `Q3 (75%)` = quantile(idade_fato, 0.75),
    Máximo = max(idade_fato)
  )

kable(t(resumo_idade), 
      col.names = "Valor",
      digits = 2) %>%
  kable_styling(bootstrap_options = c("striped", "hover"))


**Interpretação:**

\[Descrever os padrões observados na distribuição de idade, comentar sobre normalidade, identificar outliers se houver\]

## Análise da Variável: Tipo Penal


In [ ]:
#| label: fig-tipo-penal
#| fig-cap: "Distribuição de frequência do tipo penal"
#| fig-width: 8
#| fig-height: 6

# Tabela de frequências
freq_tipo <- dados %>%
  count(tipo_penal) %>%
  mutate(
    Percentual = (n / sum(n)) * 100,
    `Percentual Acumulado` = cumsum(Percentual)
  ) %>%
  arrange(desc(n))

# Gráfico de barras
ggplot(freq_tipo, aes(x = reorder(tipo_penal, -n), y = n, fill = tipo_penal)) +
  geom_col(alpha = 0.8, show.legend = FALSE) +
  geom_text(aes(label = paste0(n, "
(", round(Percentual, 1), "%)")),
            vjust = -0.5, size = 3.5) +
  labs(title = "Distribuição por Tipo Penal",
       x = "Tipo Penal",
       y = "Frequência") +
  theme_minimal() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1))


### Tabela de Frequências: Tipo Penal


In [ ]:
#| label: tbl-freq-tipo-penal
#| tbl-cap: "Frequências do tipo penal"

kable(freq_tipo, 
      col.names = c("Tipo Penal", "Frequência", "Percentual (%)", "Acumulado (%)"),
      digits = 2) %>%
  kable_styling(bootstrap_options = c("striped", "hover"))


**Interpretação:**

\[Descrever os padrões observados na distribuição do tipo penal, identificar categorias mais e menos frequentes\]

## Análise das Outras Variáveis Categóricas


In [ ]:
#| label: fig-categoricas
#| fig-cap: "Distribuição das variáveis categóricas"
#| fig-width: 10
#| fig-height: 4

p5 <- dados %>%
  count(tipo_acao) %>%
  ggplot(aes(x = tipo_acao, y = n, fill = tipo_acao)) +
  geom_col(alpha = 0.8, show.legend = FALSE) +
  geom_text(aes(label = n), vjust = -0.5) +
  labs(title = "Tipo de Ação",
       x = NULL, y = "Frequência") +
  theme_minimal()

p6 <- dados %>%
  count(reincidencia) %>%
  ggplot(aes(x = reincidencia, y = n, fill = reincidencia)) +
  geom_col(alpha = 0.8, show.legend = FALSE) +
  geom_text(aes(label = n), vjust = -0.5) +
  labs(title = "Reincidência",
       x = NULL, y = "Frequência") +
  theme_minimal()

p5 + p6


**Interpretação:**

\[Descrever os padrões observados nas variáveis tipo_acao e reincidencia\]

# Análise Exploratória de Dados (AED) - Bivariada

## Duas Variáveis Quantitativas: Dias Preso × Idade no Fato


In [ ]:
#| label: fig-scatter-dias-idade
#| fig-cap: "Relação entre dias de prisão e idade no fato"
#| fig-width: 8
#| fig-height: 6

ggplot(dados, aes(x = idade_fato, y = dias_preso)) +
  geom_point(alpha = 0.6, color = "steelblue", size = 3) +
  geom_smooth(method = "lm", se = TRUE, color = "red", linewidth = 1) +
  labs(title = "Relação entre Idade no Fato e Dias de Prisão",
       x = "Idade no fato (anos)",
       y = "Dias preso") +
  theme_minimal()


### Modelo de Regressão Linear


In [ ]:
#| label: modelo-regressao

# Ajustar modelo de regressão linear
modelo_lm <- lm(dias_preso ~ idade_fato, data = dados)

# Resumo do modelo
summary(modelo_lm)

# Coeficientes
coef_modelo <- coef(modelo_lm)


**Equação da reta:**

$$\hat{y} = `r round(coef_modelo[1], 2)` + `r round(coef_modelo[2], 2)` \cdot x$$

### Medidas de Correlação


In [ ]:
#| label: correlacao

# Correlação de Pearson
cor_pearson <- cor(dados$idade_fato, dados$dias_preso)

# Teste de correlação
teste_cor <- cor.test(dados$idade_fato, dados$dias_preso)

cat("Correlação de Pearson:", round(cor_pearson, 4), "
")
cat("Valor-p:", format.pval(teste_cor$p.value), "
")
cat("IC 95%: [", round(teste_cor$conf.int[1], 4), ",", 
    round(teste_cor$conf.int[2], 4), "]
")


**Interpretação:**

\[Interpretar a força e direção da correlação, significância estatística, e a qualidade do ajuste do modelo linear\]

## Duas Variáveis Categóricas: Tipo Penal × Reincidência


In [ ]:
#| label: tbl-contingencia
#| tbl-cap: "Tabela de contingência: Tipo Penal × Reincidência"

# Criar tabela de contingência
tabela_cont <- table(dados$tipo_penal, dados$reincidencia)

# Adicionar totais marginais
tabela_completa <- addmargins(tabela_cont)

# Calcular percentuais
tabela_prop <- prop.table(tabela_cont) * 100

kable(tabela_completa, 
      align = "c") %>%
  kable_styling(bootstrap_options = c("striped", "hover")) %>%
  add_header_above(c(" " = 1, "Reincidência" = 2, " " = 1))


### Percentuais por Linha


In [ ]:
#| label: tbl-percentuais-linha

# Percentuais por linha
tabela_prop_linha <- prop.table(tabela_cont, margin = 1) * 100

kable(round(tabela_prop_linha, 2), 
      align = "c",
      caption = "Percentuais por tipo penal (linha)") %>%
  kable_styling(bootstrap_options = c("striped", "hover")) %>%
  add_header_above(c(" " = 1, "Reincidência (%)" = 2))


### Gráfico de Barras Agrupadas


In [ ]:
#| label: fig-barras-agrupadas
#| fig-cap: "Distribuição de reincidência por tipo penal"
#| fig-width: 10
#| fig-height: 6

dados %>%
  count(tipo_penal, reincidencia) %>%
  ggplot(aes(x = tipo_penal, y = n, fill = reincidencia)) +
  geom_col(position = "dodge", alpha = 0.8) +
  geom_text(aes(label = n), 
            position = position_dodge(width = 0.9),
            vjust = -0.5, size = 3) +
  labs(title = "Reincidência por Tipo Penal",
       x = "Tipo Penal",
       y = "Frequência",
       fill = "Reincidência") +
  theme_minimal() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1))


**Interpretação:**

\[Descrever os padrões de associação entre tipo penal e reincidência\]

## Uma Variável Quantitativa e Uma Categórica: Dias Preso × Reincidência


In [ ]:
#| label: fig-boxplot-dias-reincidencia
#| fig-cap: "Comparação de dias de prisão por reincidência"
#| fig-width: 10
#| fig-height: 6

p7 <- ggplot(dados, aes(x = reincidencia, y = dias_preso, fill = reincidencia)) +
  geom_boxplot(alpha = 0.7, show.legend = FALSE) +
  labs(title = "Boxplot: Dias de Prisão por Reincidência",
       x = "Reincidência",
       y = "Dias preso") +
  theme_minimal()

p8 <- ggplot(dados, aes(x = reincidencia, y = dias_preso, fill = reincidencia)) +
  geom_violin(alpha = 0.7, show.legend = FALSE) +
  geom_boxplot(width = 0.2, alpha = 0.5, show.legend = FALSE) +
  labs(title = "Violin Plot: Dias de Prisão por Reincidência",
       x = "Reincidência",
       y = "Dias preso") +
  theme_minimal()

p7 + p8


### Medidas Resumo por Grupo


In [ ]:
#| label: tbl-resumo-grupo

resumo_grupos <- dados %>%
  group_by(reincidencia) %>%
  summarise(
    N = n(),
    Média = mean(dias_preso),
    Mediana = median(dias_preso),
    `Desvio Padrão` = sd(dias_preso),
    Mínimo = min(dias_preso),
    Máximo = max(dias_preso)
  )

kable(resumo_grupos, digits = 2) %>%
  kable_styling(bootstrap_options = c("striped", "hover"))


**Interpretação:**

\[Comparar as distribuições de dias de prisão entre reincidentes e não-reincidentes, identificar diferenças e valores influentes\]

# Reflexão sobre Variáveis Ocultas ou de Confundimento

**Possíveis variáveis de confundimento:**

1.  **Condição Socioeconômica**: Pode influenciar tanto o tipo de crime quanto o tempo de prisão, além de afetar o acesso à defesa adequada.

2.  **Região/Comarca**: Diferentes comarcas podem ter padrões distintos de aplicação de penas e concessão de liberdade.

3.  **Qualidade da Defesa**: Réus com advogados particulares vs. defensoria pública podem ter resultados diferentes.

4.  **Gravidade do Delito**: Dentro de cada tipo penal, há variações de gravidade que não foram capturadas.

5.  **Antecedentes Criminais**: Além da reincidência binária, o número e tipo de antecedentes podem influenciar.

6.  **Tempo até o Julgamento**: Pode afetar o tempo total de prisão independentemente da sentença.

\[Adicionar mais reflexões específicas ao seu contexto de pesquisa\]

# Testes de Significância contra Hipótese Nula

## Teste 1: Normalidade da Distribuição de Dias Preso

### Hipóteses

-   $H_0$: A distribuição de dias preso segue uma distribuição normal
-   $H_a$: A distribuição de dias preso não segue uma distribuição normal

**Tipo de teste:** Bilateral**Nível de significância:** $lpha = 0.05$


In [ ]:
#| label: teste-normalidade

# Teste de Shapiro-Wilk
teste_shapiro <- shapiro.test(dados$dias_preso)

cat("Teste de Shapiro-Wilk
")
cat("Estatística W:", round(teste_shapiro$statistic, 4), "
")
cat("Valor-p:", format.pval(teste_shapiro$p.value), "
")
cat("
Conclusão:")
if(teste_shapiro$p.value < 0.05) {
  cat(" Rejeita-se H0. Evidências de não-normalidade.
")
} else {
  cat(" Não se rejeita H0. Dados consistentes com normalidade.
")
}


**Interpretação:**

\[Interpretar o resultado no contexto estatístico e no contexto da pesquisa\]

## Teste 2: Correlação entre Idade no Fato e Dias Preso

### Hipóteses

-   $H_0$: $ho = 0$ (não há correlação linear entre idade no fato e dias preso)
-   $H_a$: $ho 
eq 0$ (há correlação linear entre as variáveis)

**Tipo de teste:** Bilateral**Nível de significância:** $lpha = 0.05$


In [ ]:
#| label: teste-correlacao

# Teste de correlação (já realizado anteriormente)
print(teste_cor)

cat("
Conclusão:")
if(teste_cor$p.value < 0.05) {
  cat(" Rejeita-se H0. Há evidência de correlação significativa.
")
} else {
  cat(" Não se rejeita H0. Não há evidência de correlação significativa.
")
}


**Interpretação:**

\[Interpretar o resultado no contexto estatístico e no contexto da pesquisa\]

## Teste 3: Independência entre Tipo Penal e Reincidência

### Hipóteses

-   $H_0$: Tipo penal e reincidência são independentes
-   $H_a$: Existe associação entre tipo penal e reincidência

**Tipo de teste:** Bilateral**Nível de significância:** $lpha = 0.05$


In [ ]:
#| label: teste-qui-quadrado

# Teste qui-quadrado de independência
teste_qui <- chisq.test(tabela_cont)

cat("Teste Qui-Quadrado de Independência
")
cat("X² =", round(teste_qui$statistic, 4), "
")
cat("gl =", teste_qui$parameter, "
")
cat("Valor-p:", format.pval(teste_qui$p.value), "
")

cat("
Valores Esperados:
")
print(round(teste_qui$expected, 2))

cat("
Resíduos Padronizados:
")
print(round(teste_qui$stdres, 2))

cat("
Conclusão:")
if(teste_qui$p.value < 0.05) {
  cat(" Rejeita-se H0. Há evidência de associação significativa.
")
} else {
  cat(" Não se rejeita H0. Não há evidência de associação significativa.
")
}


**Interpretação:**

\[Interpretar o resultado no contexto estatístico e no contexto da pesquisa\]

## Teste 4: Comparação de Médias - Dias Preso por Reincidência

### Hipóteses

-   $H_0$: $\mu_{	ext{Sim}} = \mu_{	ext{Não}}$ (médias iguais de dias preso para reincidentes e não-reincidentes)
-   $H_a$: $\mu_{	ext{Sim}} 
eq \mu_{	ext{Não}}$ (médias diferentes)

**Tipo de teste:** Bilateral**Nível de significância:** $lpha = 0.05$


In [ ]:
#| label: teste-t

# Teste t para duas amostras independentes
teste_t <- t.test(dias_preso ~ reincidencia, data = dados)

cat("Teste t de Student para Duas Amostras
")
cat("t =", round(teste_t$statistic, 4), "
")
cat("gl =", round(teste_t$parameter, 2), "
")
cat("Valor-p:", format.pval(teste_t$p.value), "
")
cat("IC 95% para diferença:", round(teste_t$conf.int[1], 2), "a", 
    round(teste_t$conf.int[2], 2), "
")

cat("
Conclusão:")
if(teste_t$p.value < 0.05) {
  cat(" Rejeita-se H0. Há evidência de diferença significativa nas médias.
")
} else {
  cat(" Não se rejeita H0. Não há evidência de diferença significativa nas médias.
")
}


**Interpretação:**

\[Interpretar o resultado no contexto estatístico e no contexto da pesquisa\]

# Cálculo do Tamanho da Amostra

## Para Variável Quantitativa (Dias Preso)

Usando o teste t para duas médias independentes.


In [ ]:
#| label: calculo-amostra-quantitativa

# Parâmetros
alpha <- 0.05
poder_desejado <- 0.85

# Calcular tamanho do efeito observado (d de Cohen)
media_sim <- mean(dados$dias_preso[dados$reincidencia == "Sim"])
media_nao <- mean(dados$dias_preso[dados$reincidencia == "Não"])
dp_pooled <- sqrt(((sum(dados$reincidencia == "Sim") - 1) * sd(dados$dias_preso[dados$reincidencia == "Sim"])^2 + 
                    (sum(dados$reincidencia == "Não") - 1) * sd(dados$dias_preso[dados$reincidencia == "Não"])^2) /
                   (nrow(dados) - 2))

d_cohen <- abs(media_sim - media_nao) / dp_pooled

cat("Tamanho do efeito observado (d de Cohen):", round(d_cohen, 3), "
")

# Calcular tamanho de amostra para diferentes tamanhos de efeito
tamanhos_efeito <- c(0.2, 0.5, 0.8, d_cohen)
poderes <- c(0.80, 0.85, 0.90, 0.95)

resultados_quant <- expand.grid(
  `Tamanho Efeito (d)` = tamanhos_efeito,
  `Poder (1-β)` = poderes
) %>%
  rowwise() %>%
  mutate(
    `N por Grupo` = ceiling(pwr.t.test(
      d = `Tamanho Efeito (d)`,
      sig.level = alpha,
      power = `Poder (1-β)`,
      type = "two.sample"
    )$n),
    `N Total` = `N por Grupo` * 2
  ) %>%
  arrange(`Poder (1-β)`, `Tamanho Efeito (d)`)

kable(resultados_quant, digits = 3) %>%
  kable_styling(bootstrap_options = c("striped", "hover")) %>%
  add_header_above(c(" " = 2, "Tamanho da Amostra" = 2))


## Para Variável Categórica (Reincidência)

Usando o teste qui-quadrado para proporções.


In [ ]:
#| label: calculo-amostra-categorica

# Calcular tamanho do efeito observado (w de Cohen)
# w = sqrt(sum((p_obs - p_esp)^2 / p_esp))

prop_obs <- prop.table(table(dados$tipo_penal, dados$reincidencia))
qui_obs <- chisq.test(table(dados$tipo_penal, dados$reincidencia))
w_cohen <- sqrt(qui_obs$statistic / sum(table(dados$tipo_penal, dados$reincidencia)))

cat("Tamanho do efeito observado (w de Cohen):", round(w_cohen, 3), "
")

# Calcular tamanho de amostra para diferentes tamanhos de efeito
# w: 0.1 (pequeno), 0.3 (médio), 0.5 (grande)
tamanhos_efeito_w <- c(0.1, 0.3, 0.5, as.numeric(w_cohen))

resultados_cat <- expand.grid(
  `Tamanho Efeito (w)` = tamanhos_efeito_w,
  `Poder (1-β)` = poderes
) %>%
  rowwise() %>%
  mutate(
    `N Total` = ceiling(pwr.chisq.test(
      w = `Tamanho Efeito (w)`,
      N = NULL,
      df = (5 - 1) * (2 - 1),  # (linhas - 1) * (colunas - 1)
      sig.level = alpha,
      power = `Poder (1-β)`
    )$N)
  ) %>%
  arrange(`Poder (1-β)`, `Tamanho Efeito (w)`)

kable(resultados_cat, digits = 3) %>%
  kable_styling(bootstrap_options = c("striped", "hover"))


## Recomendação Final


In [ ]:
#| label: recomendacao-amostra

# Considerar o maior N necessário entre os dois testes
n_recomendado_quant <- resultados_quant %>%
  filter(`Poder (1-β)` == 0.85, abs(`Tamanho Efeito (d)` - 0.5) < 0.01) %>%
  pull(`N Total`)

n_recomendado_cat <- resultados_cat %>%
  filter(`Poder (1-β)` == 0.85, abs(`Tamanho Efeito (w)` - 0.3) < 0.01) %>%
  pull(`N Total`)

n_final <- max(n_recomendado_quant, n_recomendado_cat, na.rm = TRUE)

cat("RECOMENDAÇÃO DE TAMANHO AMOSTRAL
")
cat("=================================

")
cat("Considerando:
")
cat("  - Erro Tipo I (α) = 5%
")
cat("  - Poder mínimo (1-β) = 85%
")
cat("  - Tamanho do efeito médio

")
cat("Tamanho de amostra recomendado:", n_final, "observações

")
cat("Para maior segurança, considerar adicionar 10-15% para compensar
")
cat("possíveis perdas e não-respostas.
")
cat("Tamanho final sugerido:", ceiling(n_final * 1.15), "observações
")


# Conclusão

\[Escrever conclusão abrangente que inclua:\]

1.  Principais achados da análise exploratória univariada
2.  Principais achados da análise exploratória bivariada
3.  Resultados dos testes de hipótese realizados
4.  Implicações dos resultados para a pesquisa
5.  Limitações do estudo atual
6.  Recomendações para a coleta de dados definitiva
7.  Próximos passos na pesquisa

# Sugestões para Melhoria da Disciplina CDEaDPP

## Aspectos Positivos

\[Listar aspectos que funcionaram bem\]

## Sugestões de Melhoria

\[Listar sugestões construtivas para melhorar a disciplina, como:\]

1.  Conteúdo programático
2.  Metodologia de ensino
3.  Atividades práticas
4.  Materiais didáticos
5.  Avaliações
6.  Recursos computacionais
7.  Outras sugestões

# Referências Bibliográficas {.unnumbered}

BUSSAB, W. O.; MORETTIN, P. A. **Estatística básica**. 9. ed. São Paulo: Saraiva, 2017.

COHEN, J. **Statistical power analysis for the behavioral sciences**. 2nd ed. Hillsdale: Lawrence Erlbaum Associates, 1988.

FIELD, A.; MILES, J.; FIELD, Z. **Descobrindo a estatística usando R**. Porto Alegre: Bookman, 2012.

髫, A.; MCCULLAGH, P. **Generalized Linear Models**. 2nd ed. London: Chapman and Hall, 1989.

MAGALHÃES, M. N.; LIMA, A. C. P. **Noções de probabilidade e estatística**. 7. ed. São Paulo: EDUSP, 2015.

MOORE, D. S.; NOTZ, W. I.; FLIGNER, M. A. **A Estatística Básica e sua prática**. 9. ed. Rio de Janeiro: LTC, 2023.

NAVARRO, D. J.; FOXCROFT, D. R. **Learning statistics with R: a tutorial for psychology students and other beginners**. Adelaide: University of Adelaide, 2019. Disponível em: https://learningstatisticswithr.com/. Acesso em: 12 dez. 2025.

POLDRACK, R. A. **Statistical thinking for the 21st century**. 2022. Disponível em: https://statsthinking21.github.io/statsthinking21-core-site/. Acesso em: 12 dez. 2025.

R CORE TEAM. **R: A language and environment for statistical computing**. Vienna: R Foundation for Statistical Computing, 2025. Disponível em: https://www.R-project.org/. Acesso em: 12 dez. 2025.

SHAPIRO, S. S.; WILK, M. B. An analysis of variance test for normality (complete samples). **Biometrika**, v. 52, n. 3/4, p. 591-611, 1965.

STUDENT. The probable error of a mean. **Biometrika**, v. 6, n. 1, p. 1-25, 1908.

TABACHNICK, B. G.; FIDELL, L. S. **Using multivariate statistics**. 7th ed. New York: Pearson, 2019.

WICKHAM, H. et al. **Welcome to the tidyverse**. Journal of Open Source Software, v. 4, n. 43, p. 1686, 2019. DOI: 10.21105/joss.01686.

WICKHAM, H.; GROLEMUND, G. **R for data science: import, tidy, transform, visualize, and model data**. 2nd ed. Sebastopol: O'Reilly Media, 2023.

::: callout-note
## Nota sobre as Referências

As referências acima incluem as principais obras e pacotes utilizados nesta análise. Adicione outras referências específicas citadas no seu trabalho, mantendo a formatação ABNT (NBR 6023:2018).
:::

------------------------------------------------------------------------

**Data de conclusão:** `r format(Sys.Date(), "%d/%m/%Y")`
